## Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API

### Objective:
Build a reusable and production-ready machine learning pipeline for predicting customer churn.

### Dataset: Telco Churn Dataset

### Instructions:
* Implement data preprocessing steps (e.g., scaling, encoding) using Pipeline
* Train models like Logistic Regression and Random Forest
* Use GridSearchCV for hyperparameter tuning
* Export the complete pipeline using joblib

In [ ]:
# Install necessary libraries
!pip install ucimlrepo scikit-learn joblib pandas numpy

### Load the Telco Churn Dataset

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Fetch dataset
telco_customer_churn = fetch_ucirepo(id=891)

# Data (as pandas dataframes)
X = telco_customer_churn.data.features
y = telco_customer_churn.data.targets

# Concatenate features and target for initial exploration
df = pd.concat([X, y], axis=1)

# Display the first 5 rows of the dataset
print("Dataset loaded successfully. First 5 rows:")
display(df.head())

### Initial Data Exploration

In [ ]:
# Display dataset information
print("\nDataset Information:")
df.info()

# Display descriptive statistics
print("\nDescriptive Statistics:")
display(df.describe(include='all'))

# Check for missing values
print("\nMissing Values:")
display(df.isnull().sum()[df.isnull().sum() > 0])

### Data Cleaning and Preprocessing

In [ ]:
# Convert 'TotalCharges' to numeric, coercing errors (empty strings) to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing 'TotalCharges' with the median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Convert 'Churn' target variable to binary (0/1)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Verify changes
print("\nUpdated Dataset Information after cleaning TotalCharges and Churn:")
df.info()

print("\nMissing values after cleaning:")
display(df.isnull().sum()[df.isnull().sum() > 0])

### Separate Features and Target, Identify Column Types

In [ ]:
# Redefine features (X) and target (y) after cleaning
X = df.drop('Churn', axis=1)
y = df['Churn']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Drop customerID if it exists, as it's not a feature for modeling
if 'customerID' in categorical_cols:
    categorical_cols.remove('customerID')
    X = X.drop('customerID', axis=1)

print("\nCategorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

### Split Data into Training and Testing Sets

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

### Create Preprocessing Pipelines

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

print("Preprocessing pipelines for numerical and categorical features created.")

### Build and Train ML Pipelines (Logistic Regression and Random Forest)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Create the Logistic Regression pipeline
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

# Create the Random Forest pipeline
random_forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Logistic Regression and Random Forest pipelines created.")

### Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid for Logistic Regression
logistic_param_grid = {
    'classifier__C': [0.1, 1.0, 10.0],
    'classifier__penalty': ['l1', 'l2']
}

# Define parameter grid for Random Forest
random_forest_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_leaf': [1, 2]
}

print("Hyperparameter grids defined for Logistic Regression and Random Forest.")

In [ ]:
# Perform GridSearchCV for Logistic Regression
logistic_grid_search = GridSearchCV(
    logistic_pipeline,
    param_grid=logistic_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Fitting Logistic Regression GridSearchCV...")
logistic_grid_search.fit(X_train, y_train)

print("Best parameters for Logistic Regression:", logistic_grid_search.best_params_)
print("Best cross-validation accuracy for Logistic Regression:", logistic_grid_search.best_score_)


In [ ]:
# Perform GridSearchCV for Random Forest
random_forest_grid_search = GridSearchCV(
    random_forest_pipeline,
    param_grid=random_forest_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Fitting Random Forest GridSearchCV...")
random_forest_grid_search.fit(X_train, y_train)

print("Best parameters for Random Forest:", random_forest_grid_search.best_params_)
print("Best cross-validation accuracy for Random Forest:", random_forest_grid_search.best_score_)

### Evaluate Models on the Test Set

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Evaluate Best Logistic Regression Model
print("\n--- Logistic Regression Model Evaluation ---")
log_reg_predictions = logistic_grid_search.best_estimator_.predict(X_test)
print("Accuracy:", accuracy_score(y_test, log_reg_predictions))
print("Classification Report:\n", classification_report(y_test, log_reg_predictions))

# Evaluate Best Random Forest Model
print("\n--- Random Forest Model Evaluation ---")
rf_predictions = random_forest_grid_search.best_estimator_.predict(X_test)
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print("Classification Report:\n", classification_report(y_test, rf_predictions))

### Export the Best Performing Pipeline

In [ ]:
import joblib

# Determine the best model based on cross-validation accuracy
if logistic_grid_search.best_score_ > random_forest_grid_search.best_score_:
    best_pipeline = logistic_grid_search.best_estimator_
    model_name = "Logistic_Regression"
else:
    best_pipeline = random_forest_grid_search.best_estimator_
    model_name = "Random_Forest"

# Export the best pipeline
joblib.dump(best_pipeline, f'best_churn_pipeline_{model_name}.joblib')

print(f"Best pipeline ({model_name}) exported successfully as best_churn_pipeline_{model_name}.joblib")

## Summary and Insights

**Summary of the ML Pipeline:**
1.  **Data Loading and Initial Exploration:** The Telco Churn dataset was fetched using `ucimlrepo`, loaded into a Pandas DataFrame, and an initial overview was performed, including checking for missing values and data types.
2.  **Data Cleaning and Preprocessing:** The `TotalCharges` column was converted to numeric (handling empty strings as NaN) and missing values were imputed using the median. The `Churn` target variable was converted to a binary format (0/1). Customer ID was dropped as it's not a predictive feature.
3.  **Feature and Target Separation:** The dataset was split into features (X) and target (y), and columns were categorized as numerical or categorical.
4.  **Data Splitting:** The data was split into training and testing sets using `train_test_split` with stratification to maintain the target variable distribution.
5.  **Preprocessing Pipelines:** `ColumnTransformer` was used to create separate preprocessing steps for numerical features (StandardScaler) and categorical features (OneHotEncoder).
6.  **Model Pipelines:** Two machine learning pipelines were built using `sklearn.pipeline.Pipeline`, one for Logistic Regression and another for Random Forest, each incorporating the `preprocessor` and a classifier.
7.  **Hyperparameter Tuning:** `GridSearchCV` was employed for both models to find the best hyperparameters using a defined parameter grid and cross-validation.
8.  **Model Evaluation:** The best models from GridSearchCV were evaluated on the unseen test set using accuracy and a classification report.
9.  **Pipeline Export:** The best-performing pipeline (either Logistic Regression or Random Forest, based on cross-validation accuracy) was exported using `joblib` for future use.

**Key Insights:**
*   The use of `Pipeline` and `ColumnTransformer` creates a robust and reproducible workflow, allowing all preprocessing steps and the model to be treated as a single unit.
*   `GridSearchCV` systematically explores hyperparameter combinations, leading to potentially better model performance than default parameters.
*   Evaluating models on a separate test set provides an unbiased estimate of the model's performance on new, unseen data.
*   Exporting the complete pipeline ensures that the exact preprocessing steps and model configuration can be reused consistently in production or for future predictions.